# NeuroMappers — Exploration Notebook

**Project:** DBS Electrode Precise Localization in the Brain  
**Course:** Biomedical Image Analysis (BMIA), University of Santiago de Compostela  
**Team:** Samad Ullah (USC) and Catarina Souto (University of Porto) — supervised by Prof. João Paulo Cunha

---

## Sprint 1 deliverables

1. **Data inspection** — verify MRI and CT format, dimensions, voxel spacing, orientation, and compatibility.
2. **Data loading** — implement functions to load NIfTI MRI and CT volumes into Python for downstream processing.

The library code lives in `src/`. This notebook demonstrates the functions against the Lead-Tutor dataset.

## Dataset

**Lead-Tutor** — Madan et al., *Aperture Neuro* (2025), DOI [10.52294/001c.129658](https://doi.org/10.52294/001c.129658). Ten patients bilaterally implanted with DBS electrodes at Brigham & Women's Hospital.

The OSF archive contains **raw native-space NIfTI only** (31 files total) organised as BIDS under `rawdata/sub-XXXXX/ses-{preop,postop}/anat/`. No bundled registration, no MNI-normalized images, and no expert electrode coordinates ship with the download.

## Setup

Make `src/` importable and load the Sprint 1 helpers.

In [1]:
import sys
from pathlib import Path


def find_project_root(marker: str = "requirements.txt") -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(f"Could not locate project root (no {marker} found above {here})")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import LEADTUTOR_DIR, PATIENT_IDS
from src.load_and_inspect import (
    load_volume,
    inspect_volume,
    inspect_many,
    find_nifti_files,
)

print(f"Project root:    {project_root}")
print(f"Lead-Tutor root: {LEADTUTOR_DIR}")
print(f"Dataset exists:  {LEADTUTOR_DIR.exists()}")

ModuleNotFoundError: No module named 'nibabel'

## 1. Discover NIfTI files

In [ ]:
raw_root = LEADTUTOR_DIR / "rawdata"
nifti_paths = find_nifti_files(raw_root)

print(f"Found {len(nifti_paths)} NIfTI files across {len(PATIENT_IDS)} subjects\n")
for p in nifti_paths[:6]:
    print(p.relative_to(LEADTUTOR_DIR))
print("...")

## 2. Inspect all volumes — format, dimensions, spacing, orientation

Runs `inspect_volume()` against every NIfTI, collects metadata into a DataFrame, and saves the result to `results/sprint1_data_inspection.csv`.

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

df = inspect_many(nifti_paths)
df["subject"] = df["path"].str.extract(r"sub-(\d+)")
df["session"] = df["path"].str.extract(r"ses-(preop|postop)")
df = df[[
    "subject", "session", "modality_guess", "format", "shape",
    "voxel_spacing_mm", "orientation", "dtype",
    "min_intensity", "max_intensity", "nan_count",
]]

output_csv = project_root / "results" / "sprint1_data_inspection.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"Saved inspection summary to {output_csv}\n")
df

## 3. Key findings

In [ ]:
print("Unique formats:", df["format"].unique())
print("\nModalities found:", sorted(df["modality_guess"].unique()))
print("\nModality counts:")
print(df["modality_guess"].value_counts().to_string())
print("\nUnique orientations across dataset:", sorted({tuple(o) for o in df["orientation"]}))
print("\nFiles containing NaN voxels:")
nan_rows = df[df["nan_count"] > 0][["subject", "session", "modality_guess", "shape", "nan_count"]]
print(nan_rows.to_string(index=False) if not nan_rows.empty else "  none")

### Observations

- **Format:** all 31 volumes are NIfTI-1 (`.nii.gz`).
- **Orientations vary:** four different axis-code triplets appear across subjects. A reorientation step to a common reference frame will be required before MRI–CT registration.
- **Voxel-spacing heterogeneity:** CTs are ~0.4–0.5 mm in-plane with 0.5–2.0 mm slice thickness; T1w usually isotropic ~1 mm; T2w anisotropic with 1.2–3.0 mm slices.
- **Intensity ranges:** CT values sit in a consistent Hounsfield range `[-1024, 3071]`; MRI intensities vary widely between subjects (no inter-subject normalization yet).
- **Discrepancies with the published dataset description:**
  - sub-39468 is listed in the PDF table as having FLAIR; the actual file is **SWI** (Susceptibility-Weighted Imaging).
  - sub-39468's post-operative CT contains ~7.45 M **NaN voxels** (~21% of the volume) — data-quality issue to raise with the supervisor.
- **MRI vs CT — main technical differences:**
  - CT is always 512×512 axial; MRI is typically 256×256 (about 4× fewer in-plane voxels).
  - CT has the finer in-plane resolution (~0.45 mm vs ~1 mm) but MRI is often isotropic.
  - CT intensity semantics are standardised (Hounsfield Units); MRI intensities are subject- and scanner-dependent.

## 4. Load and visualize a sample MRI / CT pair

Demonstrates the `load_volume()` function from `src.load_and_inspect` returning a NumPy array ready for downstream processing, together with the NIfTI affine matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_subject = "15454"
mri_path = raw_root / f"sub-{sample_subject}" / "ses-preop"  / "anat" / f"sub-{sample_subject}_ses-preop_acq-iso_T1w.nii.gz"
ct_path  = raw_root / f"sub-{sample_subject}" / "ses-postop" / "anat" / f"sub-{sample_subject}_ses-postop_CT.nii.gz"

mri_data, mri_affine, mri_img = load_volume(mri_path)
ct_data,  ct_affine,  ct_img  = load_volume(ct_path)

print(f"MRI T1w  shape={mri_data.shape}  spacing={tuple(round(z, 3) for z in mri_img.header.get_zooms()[:3])} mm")
print(f"CT       shape={ct_data.shape}  spacing={tuple(round(z, 3) for z in ct_img.header.get_zooms()[:3])} mm")

mri_mid = mri_data.shape[2] // 2
ct_mid  = ct_data.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.rot90(mri_data[:, :, mri_mid]), cmap="gray")
axes[0].set_title(f"sub-{sample_subject} · pre-op T1w · axial slice {mri_mid}")
axes[0].axis("off")

axes[1].imshow(np.rot90(ct_data[:, :, ct_mid]), cmap="gray", vmin=-100, vmax=300)
axes[1].set_title(f"sub-{sample_subject} · post-op CT · axial slice {ct_mid}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Sprint 1 status — acceptance criteria

**Task 1 — Data inspection**

- [x] MRI file format identified (NIfTI-1)
- [x] CT file format identified (NIfTI-1)
- [x] Image dimensions recorded for both datasets
- [x] Voxel spacing recorded for both datasets
- [x] Orientation / affine information checked
- [x] Main differences between MRI and CT documented
- [x] Short summary produced (`results/sprint1_data_inspection.csv` + observations above)

**Task 2 — MRI and CT loading**

- [x] MRI loads correctly (`load_volume()` in `src.load_and_inspect`)
- [x] CT loads correctly (same function)
- [x] Files can be read without errors
- [x] Volumes available as NumPy arrays with affine matrices, ready for visualization and processing

**Open questions for Prof. Cunha (to raise in class):** registration scope, source of expert electrode coordinates, handling of sub-39468 NaN issue, patient/target cohort scope — see Jira comment.